# BB8 — Phase 2 & 3: Attention Visualization

This notebook visualises the attention mechanism in BB8:

1. **Scaled dot-product attention** — step by step
2. **Causal mask** — visualised as a heatmap
3. **Multi-head attention weights** from a trained model
4. **Head diversity** — different heads attend to different things

> **Prerequisites:** Run training with `train.py` before visualising attention from a trained model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from models.attention import ScaledDotProductAttention, MultiHeadAttention
from models.transformer import create_causal_mask
from models.language_model import BB8LM
from tokenizer import CharTokenizer

sns.set_style('white')
print('Imports OK  |  PyTorch', torch.__version__)

## 1. Scaled Dot-Product Attention — Step by Step

In [ ]:
import math

torch.manual_seed(42)

# Small example: seq_len=6, d_k=8
seq_len = 6
d_k = 8
tokens = list('HAMLET')

Q = torch.randn(1, 1, seq_len, d_k)
K = torch.randn(1, 1, seq_len, d_k)
V = torch.randn(1, 1, seq_len, d_k)

# Step 1: Raw scores
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

# Step 2: Softmax → attention weights
attn_weights = torch.softmax(scores, dim=-1)

# Step 3: Causal mask
mask = create_causal_mask(seq_len, device=torch.device('cpu'))
masked_scores = scores.masked_fill(mask == 0, float('-inf'))
causal_weights = torch.softmax(masked_scores, dim=-1)
causal_weights = torch.nan_to_num(causal_weights, nan=0.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.heatmap(scores.squeeze().numpy(), ax=axes[0], annot=True, fmt='.1f',
            xticklabels=tokens, yticklabels=tokens, cmap='RdBu_r', center=0)
axes[0].set_title('Raw Scores (QKᵀ / √dₖ)', fontsize=12)

sns.heatmap(attn_weights.squeeze().numpy(), ax=axes[1], annot=True, fmt='.2f',
            xticklabels=tokens, yticklabels=tokens, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Attention Weights (no mask)', fontsize=12)

sns.heatmap(causal_weights.squeeze().numpy(), ax=axes[2], annot=True, fmt='.2f',
            xticklabels=tokens, yticklabels=tokens, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title('Causal Attention Weights (masked)', fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/attention_steps.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Causal Mask Visualisation

In [ ]:
T = 12
mask = create_causal_mask(T, device=torch.device('cpu')).squeeze().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mask, cmap='Greys', vmin=0, vmax=1)

ax.set_title(f'Causal Mask (seq_len={T})\nDark = can attend, Light = blocked', fontsize=12)
ax.set_xlabel('Key position (source)')
ax.set_ylabel('Query position (target)')

# Add grid
ax.set_xticks(range(T))
ax.set_yticks(range(T))
ax.grid(color='lightgray', linewidth=0.5)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('../outputs/causal_mask.png', dpi=150)
plt.show()

print('Causal mask interpretation:')
print('  Row i, Col j = 1 (dark) → position i CAN attend to position j (j ≤ i)')
print('  Row i, Col j = 0 (white) → position i CANNOT attend to j (j > i, the future)')

## 3. Multi-Head Attention Weights from a Trained Model

> This section requires a trained checkpoint. Run `train.py` first.

In [ ]:
CHECKPOINT_PATH = '../checkpoints/run/best_model.pt'
DATA_PATH = '../data/tiny_shakespeare.txt'

if not os.path.exists(CHECKPOINT_PATH):
    print(f'Checkpoint not found at {CHECKPOINT_PATH}')
    print('Train a model first: python train.py --data data/tiny_shakespeare.txt')
else:
    # Load tokenizer
    with open(DATA_PATH, 'r') as f:
        text = f.read()
    tokenizer = CharTokenizer()
    tokenizer.train([text])

    # Load model
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
    cfg = ckpt.get('model_config', {})
    model = BB8LM(
        vocab_size=tokenizer.get_vocab_size(),
        d_model=cfg.get('d_model', 128),
        num_layers=cfg.get('num_layers', 4),
        num_heads=cfg.get('num_heads', 4),
        d_ff=cfg.get('d_ff', 512),
        max_seq_len=cfg.get('max_seq_len', 256),
        dropout=0.0,
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print('Model loaded!')

    # Encode a prompt
    prompt = 'To be, or not to be: that is the question'
    ids = tokenizer.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long)
    
    with torch.no_grad():
        output = model(x, return_attn_weights=True)
    
    attn_weights_all = output['attn_weights']  # list of (1, heads, T, T)
    print(f'Attention weights collected from {len(attn_weights_all)} layers')

In [ ]:
# Visualise attention heads for layer 0
if 'attn_weights_all' in dir() and attn_weights_all:
    layer_idx = 0
    attn = attn_weights_all[layer_idx].squeeze(0).numpy()  # (heads, T, T)
    num_heads = attn.shape[0]
    chars = list(prompt[:attn.shape[1]])

    fig, axes = plt.subplots(1, num_heads, figsize=(4*num_heads, 4))
    if num_heads == 1:
        axes = [axes]

    for h, ax in enumerate(axes):
        sns.heatmap(
            attn[h],
            ax=ax,
            cmap='Blues',
            xticklabels=chars,
            yticklabels=chars,
            vmin=0,
            vmax=attn[h].max(),
            cbar=False,
        )
        ax.set_title(f'Layer {layer_idx+1}, Head {h+1}', fontsize=10)
        ax.tick_params(axis='x', rotation=0, labelsize=7)
        ax.tick_params(axis='y', rotation=0, labelsize=7)

    fig.suptitle(
        f'Attention Weights — Layer {layer_idx+1}\n"{prompt[:20]}..."',
        fontsize=13
    )
    plt.tight_layout()
    plt.savefig(f'../outputs/attention_layer{layer_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No attention weights available. Train a model first.')

## 4. Average Attention Distance per Head

How far back does each attention head typically look?

In [ ]:
if 'attn_weights_all' in dir() and attn_weights_all:
    T = attn_weights_all[0].shape[-1]
    positions = torch.arange(T, dtype=torch.float)
    
    avg_distances = []
    for layer_attn in attn_weights_all:
        # layer_attn: (1, heads, T, T)
        attn = layer_attn.squeeze(0)  # (heads, T, T)
        heads_distances = []
        for head in range(attn.shape[0]):
            w = attn[head]  # (T, T)
            # Expected distance for each query position
            for q in range(T):
                dist = sum(w[q, k].item() * abs(q - k) for k in range(T))
                heads_distances.append(dist)
        avg_distances.append(heads_distances)

    print('Average attention distance per layer and head (lower = more local):')
    for l, dists in enumerate(avg_distances):
        per_head = [sum(dists[h::4]) / len(dists[h::4]) for h in range(4)]
        print(f'  Layer {l+1}: {[f"{d:.1f}" for d in per_head]}')
else:
    print('No attention weights available.')